# 06 — Visualisations

Five publication-ready figures saved to `outputs/` at 150 dpi.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib_venn import venn2
import umap

OUTPUTS = '../outputs'
DPI = 150

# Colour palette
TEAL   = '#2A9D8F'
AMBER  = '#E9C46A'
CORAL  = '#E76F51'
GRAY   = '#9E9E9E'

## 1 · UMAP — JobBERT candidate embeddings by job domain

In [2]:
features      = pd.read_csv(f'{OUTPUTS}/features.csv')
candidate_vecs = np.load(f'{OUTPUTS}/candidate_vecs_jobbert.npy')
job_vecs       = np.load(f'{OUTPUTS}/job_vecs_jobbert.npy')

DOMAIN_MAP = {
    'Machine Learning (ML) Engineer':                                              'Data & AI',
    'AI Engineer':                                                                 'Data & AI',
    'Data Engineer':                                                               'Data & AI',
    'Data Science Engineer':                                                       'Data & AI',
    'Intern (Generative AI Engineering - 2D/3D Image Generation)':                'Data & AI',
    'Full Stack Developer (Python,React js)':                                     'Software Engineering',
    'Senior Software Engineer':                                                    'Software Engineering',
    'Senior iOS Engineer':                                                         'Software Engineering',
    'DevOps Engineer':                                                             'Software Engineering',
    'Network Support Engineer':                                                    'Software Engineering',
    'System Administrator (Operation & Maintenance of Server, Storage & Service Desk System)': 'Software Engineering',
    'Executive/ Sr. Executive -IT':                                               'Software Engineering',
    'Database Administrator (DBA)':                                               'Software Engineering',
    'Civil Engineer':                                                              'Engineering',
    'Mechanical Engineer':                                                         'Engineering',
    'Mechanical Designer':                                                         'Engineering',
    'Site Engineer':                                                               'Engineering',
    'Project Coordinator (Civil)':                                                'Engineering',
    'Management Trainee - Mechanical':                                             'Engineering',
    'Manager- Human Resource Management (HRM)':                                   'HR & Admin',
    'HR Officer':                                                                  'HR & Admin',
    'Asst. Manager/ Manger (Administrative)':                                     'HR & Admin',
    'Head of Internal Control & Compliance (ICC) - SEVP/DMD':                    'Finance & Compliance',
    'Sr.Officer / Executive - Internal Audit':                                    'Finance & Compliance',
    'Executive - VAT':                                                             'Finance & Compliance',
    'Business Development Executive':                                              'Sales & Marketing',
    'Executive/ Senior Executive- Trade Marketing, Hygiene Products':             'Sales & Marketing',
    'Marketing Officer':                                                           'Sales & Marketing',
}

DOMAIN_COLORS = {
    'Data & AI':            '#4C72B0',
    'Software Engineering': '#55A868',
    'Engineering':          '#C44E52',
    'HR & Admin':           '#8172B2',
    'Finance & Compliance': '#CCB974',
    'Sales & Marketing':    '#64B5CD',
}

features['domain'] = features['job_position_name'].map(DOMAIN_MAP)

# One vector per unique candidate (deduplicate on candidate_doc)
cand_df = features.drop_duplicates(subset='candidate_doc').reset_index(drop=False).rename(columns={'index': 'orig_idx'})
cand_vecs_unique = candidate_vecs[cand_df['orig_idx'].values]
cand_domains     = cand_df['domain'].values

# One vector per unique job
job_df = features.drop_duplicates(subset='job_position_name').reset_index(drop=False).rename(columns={'index': 'orig_idx'})
job_vecs_unique = job_vecs[job_df['orig_idx'].values]
job_domains     = job_df['domain'].values

# Fit UMAP on all vectors jointly so candidate and job embeddings share the same 2D space
all_vecs  = np.vstack([cand_vecs_unique, job_vecs_unique])
reducer   = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.15, random_state=42, metric='cosine')
embedding = reducer.fit_transform(all_vecs)

cand_emb = embedding[:len(cand_vecs_unique)]
job_emb  = embedding[len(cand_vecs_unique):]

fig, ax = plt.subplots(figsize=(9, 7))

for domain, color in DOMAIN_COLORS.items():
    mask = cand_domains == domain
    ax.scatter(cand_emb[mask, 0], cand_emb[mask, 1],
               c=color, s=14, alpha=0.55, linewidths=0, label=domain)

# Job vectors as star markers
for i, (domain, jname) in enumerate(zip(job_domains, job_df['job_position_name'].values)):
    color = DOMAIN_COLORS.get(domain, '#333')
    ax.scatter(job_emb[i, 0], job_emb[i, 1],
               marker='*', s=220, c=color, edgecolors='white', linewidths=0.6, zorder=5)

legend_patches = [mpatches.Patch(color=c, label=d) for d, c in DOMAIN_COLORS.items()]
ax.legend(handles=legend_patches, fontsize=8, loc='upper left',
          framealpha=0.85, edgecolor='#ccc', title='Job domain', title_fontsize=8)

ax.set_title('UMAP — JobBERT candidate embeddings by job domain\n(stars = job vectors)', fontsize=11)
ax.set_xlabel('UMAP 1', fontsize=9)
ax.set_ylabel('UMAP 2', fontsize=9)
ax.tick_params(labelsize=8)
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
path = f'{OUTPUTS}/viz_umap_embeddings.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved {path}')

/Users/nadine/tensorflow-env/env/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Saved ../outputs/viz_umap_embeddings.png


## 2 · Vocab Venn — candidate skill tokens vs job skill tokens

In [3]:
# Subsets: (left-only, right-only, intersection) = (2797-31, 97-31, 31)
CAND_TOTAL  = 2797
JOB_TOTAL   = 97
COMMON      = 31
cand_only   = CAND_TOTAL - COMMON
job_only    = JOB_TOTAL  - COMMON

fig, ax = plt.subplots(figsize=(7, 5))

v = venn2(
    subsets=(cand_only, job_only, COMMON),
    set_labels=('Candidate skills', 'Job skills'),
    set_colors=(TEAL, AMBER),
    alpha=0.65,
    ax=ax,
)

# Label styling
for lbl in ['10', '01', '11']:
    t = v.get_label_by_id(lbl)
    if t:
        t.set_fontsize(12)
        t.set_fontweight('bold')

for lbl_id, txt in [('A', 'Candidate skills'), ('B', 'Job skills')]:
    t = v.get_label_by_id(lbl_id)
    if t:
        t.set_fontsize(10)

ax.annotate(
    '2,797 candidate tokens  ·  97 job tokens  ·  31 in common',
    xy=(0.5, -0.06), xycoords='axes fraction',
    ha='center', fontsize=9, color='#444',
    style='italic',
)

ax.set_title('Skill vocabulary overlap — candidate vs job side', fontsize=11, pad=10)

fig.tight_layout()
path = f'{OUTPUTS}/viz_vocab_venn.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved {path}')

Saved ../outputs/viz_vocab_venn.png


## 3 · Feature importance — Model B permutation importances

In [4]:
features_imp = [
    ('exp_deficit',           0.130, 'exp'),
    ('st_cosine_jobbert',     0.100, 'semantic'),
    ('skills_required_count', 0.060, 'structured'),
    ('title_semantic_sim',    0.040, 'semantic'),
    ('exp_surplus',           0.020, 'exp'),
    ('skill_semantic_sim',    0.005, 'semantic'),
    ('skill_coverage',        0.002, 'structured'),
    ('edu_match',             0.002, 'structured'),
]

COLOR_MAP = {'exp': CORAL, 'semantic': TEAL, 'structured': GRAY}

labels      = [f[0] for f in features_imp]
importances = [f[1] for f in features_imp]
colors      = [COLOR_MAP[f[2]] for f in features_imp]

# Sort ascending for horizontal bar (largest at top)
order       = np.argsort(importances)
labels_s    = [labels[i] for i in order]
imports_s   = [importances[i] for i in order]
colors_s    = [colors[i] for i in order]

fig, ax = plt.subplots(figsize=(7, 4.5))

bars = ax.barh(labels_s, imports_s, color=colors_s, height=0.6, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, imports_s):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha='left', fontsize=8.5, color='#333')

legend_patches = [
    mpatches.Patch(color=CORAL, label='Experience features'),
    mpatches.Patch(color=TEAL,  label='Semantic features'),
    mpatches.Patch(color=GRAY,  label='Structured features'),
]
ax.legend(handles=legend_patches, fontsize=8.5, loc='lower right',
          framealpha=0.85, edgecolor='#ccc')

ax.set_xlabel('Permutation importance (mean decrease in R²)', fontsize=9)
ax.set_title('Model B — feature importances (HistGradientBoosting)', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(labelsize=8.5)
ax.set_xlim(0, max(imports_s) * 1.18)

fig.tight_layout()
path = f'{OUTPUTS}/viz_feature_importance.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved {path}')

Saved ../outputs/viz_feature_importance.png


## 4 · Score distribution — matched_score histogram

In [5]:
scores = features['matched_score']
mean_s = scores.mean()
med_s  = scores.median()

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(scores, bins=40, color=AMBER, edgecolor='white', linewidth=0.4, alpha=0.9)

ax.axvline(mean_s, color='#E63946', linestyle='--', linewidth=1.4, label=f'Mean {mean_s:.3f}')
ax.axvline(med_s,  color='#457B9D', linestyle=':',  linewidth=1.4, label=f'Median {med_s:.3f}')

# Annotate dominant clusters
ymax = ax.get_ylim()[1]
for val, label_txt, xoff in [(0.65, '0.65 cluster', 0.012), (0.85, '0.85 cluster', 0.012)]:
    ax.axvline(val, color='#555', linestyle='-', linewidth=0.8, alpha=0.5)
    ax.text(val + xoff, ymax * 0.88, label_txt,
            fontsize=8, color='#333', rotation=90, va='top')

ax.legend(fontsize=9, framealpha=0.85, edgecolor='#ccc')
ax.set_xlabel('matched_score', fontsize=9)
ax.set_ylabel('Count', fontsize=9)
ax.set_title('Distribution of matched_score (n = 9,460)', fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(labelsize=8.5)

fig.tight_layout()
path = f'{OUTPUTS}/viz_score_dist.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved {path}')

Saved ../outputs/viz_score_dist.png


## 5 · Predicted vs true — Model A and Model B (test set)

In [6]:
pred_a = pd.read_csv(f'{OUTPUTS}/model_a_predictions.csv')
pred_b = pd.read_csv(f'{OUTPUTS}/model_b_predictions.csv')

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)

for ax, df, pred_col, title, color in [
    (axes[0], pred_a, 'model_a_pred', 'Model A  (TF-IDF + Ridge)',         '#4C72B0'),
    (axes[1], pred_b, 'model_b_pred', 'Model B  (JobBERT + HGB)',           TEAL),
]:
    true = df['matched_score'].values
    pred = df[pred_col].values

    ax.scatter(true, pred, c=color, s=10, alpha=0.35, linewidths=0)

    lo = min(true.min(), pred.min()) - 0.02
    hi = max(true.max(), pred.max()) + 0.02
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.0, label='Perfect')

    mae  = np.mean(np.abs(true - pred))
    rmse = np.sqrt(np.mean((true - pred) ** 2))
    ax.text(0.05, 0.93, f'MAE={mae:.3f}   RMSE={rmse:.3f}',
            transform=ax.transAxes, fontsize=8.5, color='#333',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ccc', alpha=0.85))

    ax.set_xlabel('True score', fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=8)
    ax.set_aspect('equal', adjustable='box')

axes[0].set_ylabel('Predicted score', fontsize=9)
fig.suptitle('Predicted vs true score — test set (n = 2,027)', fontsize=11, y=1.02)
fig.tight_layout()

path = f'{OUTPUTS}/viz_pred_vs_true.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved {path}')

Saved ../outputs/viz_pred_vs_true.png
